# 02 Public 10x Xenium cohort: WT 2.5 / 5.7 / 13.4 mo (Figure 1) and TgCRND8 2.5 / 5.7 / 17.9 mo (Figure 4, incl. summary schematic 4F)
**Part A** builds the annotated object once (slow: six bundles, label transfer) and writes `data/10xAD_all_annotated.h5ad`.
**Part B** starts from that file: statistics, transcript-level analysis, Figures 1 and 4, source data (Supplementary Tables S3 and S5).

Fresh kernel: B0, then A1 to A3 only if the annotated object has to be rebuilt, then B1 to B8 in order.
Figures 1 and 4 are saved to `figures/panels/` and `final_data/figures/`.

v7: Figure 4 with tighter panel A and new panel 4G (quantification across all cohorts). 4G needs the tables of notebooks 01 and 05.

In [ ]:
# ==========================================================
# B0) Setup: paths, style, helpers (10x Xenium WT + TgCRND8)
# ==========================================================
import json, re, gc, warnings
import numpy as np, pandas as pd
import anndata as ad, scanpy as sc
import matplotlib as mpl, matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import StratifiedTable
from statsmodels.stats.multitest import multipletests
from scipy.spatial import cKDTree
from pathlib import Path
from PIL import Image
warnings.filterwarnings("ignore")

import os
# Project folder with data/, tables/, figures/ and final_data/ (see README).
# Default: the parent of the notebooks folder. Override with the environment variable SPP1_PROJECT_DIR.
OUT = Path(os.environ.get("SPP1_PROJECT_DIR", Path.cwd().parent)).resolve()
FIG, TAB, DAT = OUT / "figures", OUT / "tables", OUT / "data"
PANELS = FIG / "panels"; PANELS.mkdir(parents=True, exist_ok=True)
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
    "legend.fontsize": 6.5, "axes.linewidth": 0.6, "pdf.fonttype": 42, "svg.fonttype": "none",
    "savefig.dpi": 300, "figure.dpi": 150, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False})
MM = 1 / 25.4

def dense(M):
    return M.toarray() if hasattr(M, "toarray") else np.asarray(M)

FINAL = OUT / "final_data" / "figures"; FINAL.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    """Panels and drafts go to figures/panels; Fig1 to Fig3 additionally to final_data/figures."""
    for d in [PANELS] + ([FINAL] if re.fullmatch(r"Fig\d", name) else []):
        fig.savefig(d / f"{name}.pdf", bbox_inches="tight")
        fig.savefig(d / f"{name}.png", dpi=300, bbox_inches="tight")
        if d == FINAL:                                   # Frontiers: TIFF, RGB, 300 dpi, LZW
            Image.open(d / f"{name}.png").convert("RGB").save(d / f"{name}.tif", compression="tiff_lzw", dpi=(300, 300))
    plt.show()

print(sorted(p.name for p in DAT.iterdir() if p.is_dir()))

## Part A: build the annotated object (run once)

In [ ]:
# ==========================================================
# A1) Load all six 10x bundles (WT + TgCRND8), parse genotype and age from folder name
# ==========================================================
def find(folder, name):
    hits = sorted(folder.rglob(name))
    return hits[0] if hits else None

bundles = sorted({p.parent for p in DAT.rglob("cell_feature_matrix.h5")})
ads, meta = [], []
for b in bundles:
    name = b.name if b.name not in ["outs", "output"] else b.parent.name
    mt = re.search(r"FFPE_(.+?)_(\d+)_(\d+)_months", name)
    if mt is None:
        print("skipped (name not parsed):", name); continue
    geno = "WT" if "wild" in mt.group(1).lower() else "TgCRND8"
    age = float(f"{mt.group(2)}.{mt.group(3)}")
    cp = find(b, "cells.parquet") or find(b, "cells.csv.gz")
    x = sc.read_10x_h5(b / "cell_feature_matrix.h5"); x.var_names_make_unique()
    cells = pd.read_parquet(cp) if cp.suffix == ".parquet" else pd.read_csv(cp)
    cells["cell_id"] = cells["cell_id"].astype(str)
    cells = cells.set_index("cell_id").loc[x.obs_names.astype(str)]
    x.obs = cells[[c for c in ["x_centroid", "y_centroid", "cell_area"] if c in cells]].copy()
    x.obs["sample"] = name; x.obs["genotype"] = geno; x.obs["age_mo"] = age
    x.obsm["spatial"] = x.obs[["x_centroid", "y_centroid"]].to_numpy()
    x.obs_names = name + "_" + x.obs_names.astype(str)
    ads.append(x); meta.append({"sample": name, "genotype": geno, "age": age, "n_cells": x.n_obs})
print(pd.DataFrame(meta).sort_values(["genotype", "age"]).to_string(index=False))

xa = ad.concat(ads, join="inner", merge="same"); del ads; gc.collect()
xa = xa[:, ~xa.var_names.str.contains("NegControl|BLANK|Codeword|Unassigned|Deprecated|Intergenic", case=False)].copy()
xa.layers["counts"] = xa.X.copy()
xa.obs["n_counts"] = np.asarray(xa.X.sum(axis=1)).ravel()
for c in ["sample", "genotype"]: xa.obs[c] = xa.obs[c].astype(str)
xa.obs["age_mo"] = xa.obs["age_mo"].astype(float)
xa = xa[xa.obs["n_counts"] >= 20].copy()
print(xa)
print(xa.obs.groupby(["genotype", "age_mo"])["n_counts"].median())
xa.write_h5ad(DAT / "10xAD_all_raw.h5ad")

In [ ]:
# ==========================================================
# A2) Cell-type label transfer from our 247-panel sections (logistic regression on shared genes)
# Reference: the three sections of our cohort (Zenodo). The labels used in the paper were trained on these
# three sections plus one further section of the same panel; they are provided on Zenodo (public10x_celltype_labels.csv).
# ==========================================================
from sklearn.linear_model import LogisticRegression

def prep(M):
    M = dense(M).astype(np.float32)
    s = M.sum(axis=1, keepdims=True); s[s == 0] = 1
    return np.log1p(M / s * 100)

ref = sc.read_h5ad(DAT / "own_cohort_xenium_247panel.h5ad", backed="r")
genes = list(ref.var_names); lab_all = ref.obs["cell_type"].astype(str).to_numpy()
rng_ = np.random.default_rng(0)
idx = np.sort(np.concatenate([rng_.choice(np.where(lab_all == c)[0], size=min(5000, (lab_all == c).sum()), replace=False)
                              for c in np.unique(lab_all)]))
mask = np.zeros(ref.n_obs, bool); mask[idx] = True
blocks = []
for s in range(0, ref.n_obs, 50_000):
    e = min(s + 50_000, ref.n_obs); m = mask[s:e]
    if m.any(): blocks.append(prep(ref.layers["counts"][s:e])[m])
Xtr = np.vstack(blocks); ytr = lab_all[idx]; del blocks
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
clf = LogisticRegression(max_iter=300, C=0.5, n_jobs=1).fit((Xtr - mu) / sd, ytr)
del Xtr; gc.collect()

gi = [list(xa.var_names).index(g) for g in genes]
Cq = xa.layers["counts"]; lab_q, conf_q = [], []
for s in range(0, xa.n_obs, 20_000):
    P = clf.predict_proba((prep(Cq[s:s + 20_000][:, gi]) - mu) / sd)
    lab_q.append(clf.classes_[P.argmax(1)]); conf_q.append(P.max(1))
    print(f"\r  query cells {min(s + 20_000, xa.n_obs):,}/{xa.n_obs:,}", end="")
print()
xa.obs["cell_type"] = np.concatenate(lab_q); xa.obs["ct_conf"] = np.concatenate(conf_q)
xa.obs.loc[xa.obs["ct_conf"] < 0.5, "cell_type"] = "Low confidence"
print(pd.crosstab(xa.obs["cell_type"], [xa.obs["genotype"], xa.obs["age_mo"]]).to_string())
xa.write_h5ad(DAT / "10xAD_all_celltypes.h5ad")

In [ ]:
# ==========================================================
# A3) Strict microglia / astrocytes, Spp1 by cell type, local fibre context -> 10xAD_all_annotated.h5ad
# ==========================================================
def gsum(gl):
    return dense(xa[:, [g for g in gl if g in xa.var_names]].layers["counts"]).sum(axis=1)

o = xa.obs
ct, conf = o["cell_type"].to_numpy(), o["ct_conf"].to_numpy()
o["is_mg"] = (ct == "Microglia") & (conf >= 0.7) & (gsum(["P2ry12", "Tmem119", "Cx3cr1", "Gpr34"]) >= 2)
o["is_astro"] = (ct == "Astrocyte") & (conf >= 0.7) & (gsum(["Aldh1l1", "Gja1"]) >= 2)
o["Spp1_n"] = dense(xa[:, "Spp1"].layers["counts"]).ravel()

# local context: oligodendrocyte share among 30 nearest neighbours
K = 30
o["f_oligo"] = np.nan
for s in o["sample"].unique():
    m = np.where((o["sample"] == s).to_numpy())[0]; xy = xa.obsm["spatial"][m]
    _, nn = cKDTree(xy).query(xy, k=K + 1)
    o.iloc[m, o.columns.get_loc("f_oligo")] = (ct[m][nn[:, 1:]] == "Oligodendrocyte").mean(1)
o["ctx_bin"] = pd.cut(o["f_oligo"], [-0.01, 0.2, 0.4, 1.0], labels=["Grey", "Border", "Fibre"]).astype(str)

print("strict microglia / astrocytes:")
print(o.groupby(["genotype", "age_mo"])[["is_mg", "is_astro"]].sum().astype(int).to_string())

grp = np.where(o["is_mg"], "Microglia (strict)", np.where(o["is_astro"], "Astrocyte (strict)", ct))
t4 = (pd.DataFrame({"group": grp, "geno": o["genotype"], "age": o["age_mo"], "pos": o["Spp1_n"] >= 1, "n": o["Spp1_n"]})
        .groupby(["group", "geno", "age"]).agg(pct=("pos", "mean"), mean_pos=("n", lambda v: v[v >= 1].mean() if (v >= 1).any() else np.nan))
        .reset_index())
t4["pct"] *= 100
t4.to_csv(TAB / "10xAD_04_spp1_by_celltype.csv", index=False)
print("\nSpp1+ (%):")
print(t4.pivot_table(index="group", columns=["geno", "age"], values="pct").round(1)
        .loc[["Microglia (strict)", "Astrocyte (strict)", "Oligodendrocyte", "VLMC / fibroblast", "Inhibitory neuron"]].to_string())
print("\nSpp1 transcripts per positive cell:")
print(t4.pivot_table(index="group", columns=["geno", "age"], values="mean_pos").round(2)
        .loc[["Microglia (strict)", "Astrocyte (strict)"]].to_string())
xa.write_h5ad(DAT / "10xAD_all_annotated.h5ad")

## Part B: analyses, figures, source data

In [ ]:
# ==========================================================
# B1) Load the annotated object (start here unless Part A has to be redone)
# ==========================================================
if "xa" not in globals() or "is_mg" not in xa.obs:
    xa = sc.read_h5ad(DAT / "10xAD_all_annotated.h5ad")
o = xa.obs
o["age_mo"] = o["age_mo"].astype(float)
o["ctx_bin"] = pd.cut(o["f_oligo"], [-0.01, 0.2, 0.4, 1.0], labels=["Grey", "Border", "Fibre"]).astype(str)
print(o.groupby(["genotype", "age_mo"])[["is_mg", "is_astro"]].sum().astype(int).to_string())

In [ ]:
# ==========================================================
# B2) Core analyses per genotype: Spp1+ by context x age, clustering, astrocyte contact
# ==========================================================
o["d_vlmc"] = np.nan; o["n_astro25"] = np.nan
for s in o["sample"].unique():
    m = np.where((o["sample"] == s).to_numpy())[0]; xy = xa.obsm["spatial"][m]
    is_mg, is_as = o["is_mg"].to_numpy()[m], o["is_astro"].to_numpy()[m]
    vl = o["cell_type"].to_numpy()[m] == "VLMC / fibroblast"
    o.iloc[m[is_mg], o.columns.get_loc("d_vlmc")] = cKDTree(xy[vl]).query(xy[is_mg])[0]
    o.iloc[m[is_mg], o.columns.get_loc("n_astro25")] = cKDTree(xy[is_as]).query_ball_point(xy[is_mg], 25, return_length=True)

mg = o[o["is_mg"]].copy()
mg["spp1"] = (mg["Spp1_n"] >= 1).astype(int)
mg["aged"] = (mg["age_mo"] == mg.groupby("genotype")["age_mo"].transform("max")).astype(int)
mg.to_csv(TAB / "10xAD_06_strict_microglia.csv")

def ci_or(f, term):
    lo, hi = np.exp(f.conf_int().loc[term]); return f"OR {np.exp(f.params[term]):.2f} [{lo:.2f}, {hi:.2f}], p = {f.pvalues[term]:.2g}"

rng_ = np.random.default_rng(0)
summary = []
for g in ["WT", "TgCRND8"]:
    t = mg[mg["genotype"] == g]
    print(f"\n================ {g} ================")
    tab = t.groupby(["ctx_bin", "age_mo"]).agg(n=("spp1", "size"), k=("spp1", "sum")); tab["pct"] = 100 * tab["k"] / tab["n"]
    print("6a  Spp1+ microglia (%) by context x age:")
    print(tab["pct"].unstack().round(1).reindex(["Grey", "Border", "Fibre"]).to_string())
    print(tab["k"].unstack().reindex(["Grey", "Border", "Fibre"]).to_string())
    f = smf.glm("spp1 ~ aged + ctx_bin", data=t, family=sm.families.Binomial()).fit(cov_type="HC1")
    print("    oldest vs younger, context-adjusted:", ci_or(f, "aged"))

    for a_ in sorted(t["age_mo"].unique()):
        ta = t[t["age_mo"] == a_]
        if ta["spp1"].sum() < 10: continue
        # 6b clustering (within-context permutation)
        xy = ta[["x_centroid", "y_centroid"]].to_numpy(); lab_ = ta["spp1"].to_numpy(); bins_ = ta["ctx_bin"].to_numpy()
        pairs = cKDTree(xy).query_ball_point(xy, 50)
        def clust(l):
            p = np.where(l == 1)[0]; return np.mean([any(l[j] == 1 for j in pairs[i] if j != i) for i in p])
        obs_c = clust(lab_); null = []
        for _ in range(500):
            l = lab_.copy()
            for b in np.unique(bins_):
                ib = np.where(bins_ == b)[0]; l[ib] = rng_.permutation(l[ib])
            null.append(clust(l))
        p_cl = (np.sum(np.array(null) >= obs_c) + 1) / 501
        # 6c astrocyte contact
        fa = smf.glm("spp1 ~ n_astro25 + ctx_bin", data=ta, family=sm.families.Binomial()).fit(cov_type="HC1")
        tabs = [pd.crosstab(tt["spp1"], tt["n_astro25"] >= 1).reindex(index=[1, 0], columns=[True, False], fill_value=0).to_numpy()
                for _, tt in ta.groupby("ctx_bin") if tt["spp1"].sum() > 0 and (tt["spp1"] == 0).sum() > 0]
        stt = StratifiedTable(tabs)
        print(f"\n  {a_:g} mo: n Spp1+ = {lab_.sum()} | clustering {100 * obs_c:.0f}% vs null {100 * np.mean(null):.0f}% (p = {p_cl:.3f})")
        print(f"    per astrocyte neighbour: {ci_or(fa, 'n_astro25')}")
        print(f"    any astrocyte contact (CMH): OR {stt.oddsratio_pooled:.2f}, p = {stt.test_null_odds().pvalue:.3f}")
        print(ta.groupby(["ctx_bin", "spp1"])["n_astro25"].agg(["mean", "size"]).round(2).unstack().to_string())
        summary.append({"genotype": g, "age": a_, "n_mg": len(ta), "n_spp1": int(lab_.sum()),
                        "clust_obs": obs_c, "clust_null": np.mean(null), "p_clust": p_cl,
                        "OR_astro": np.exp(fa.params["n_astro25"]), "p_astro": fa.pvalues["n_astro25"],
                        "OR_cmh": stt.oddsratio_pooled, "p_cmh": stt.test_null_odds().pvalue})
pd.DataFrame(summary).to_csv(TAB / "10xAD_06_summary.csv", index=False)
print("\n", pd.DataFrame(summary).round(3).to_string(index=False))

In [ ]:
# ==========================================================
# B3) Astrocytes next to Spp1+ vs Spp1- microglia (gene detection), every animal with >= 10 Spp1+ microglia
# ==========================================================
AG = [g for g in ["Gfap", "Serpina3n", "Cd44", "Vim", "C4b", "C3", "Lcn2", "Osmr", "Aqp4", "Aldh1l1", "Gja1"] if g in xa.var_names]
NPERM = 10000; res = []
for s in o["sample"].unique():
    m = np.where((o["sample"] == s).to_numpy())[0]; xy = xa.obsm["spatial"][m]
    is_as = o["is_astro"].to_numpy()[m]; mgm = o["is_mg"].to_numpy()[m]
    pos = mgm & (o["Spp1_n"].to_numpy()[m] >= 1); neg = mgm & ~pos
    if pos.sum() < 10: continue
    a_xy = xy[is_as]
    near_pos = cKDTree(xy[pos]).query_ball_point(a_xy, 25, return_length=True) > 0
    near_neg = cKDTree(xy[neg]).query_ball_point(a_xy, 25, return_length=True) > 0
    keep = near_pos | near_neg
    lab = near_pos[keep]; bins_a = o["ctx_bin"].to_numpy()[m][is_as][keep]
    Cm = (dense(xa[m[is_as], AG].layers["counts"]) >= 1)[keep]
    if lab.sum() < 3: continue
    perms = np.empty((NPERM, len(lab)), bool)
    for i in range(NPERM):
        l = lab.copy()
        for b in np.unique(bins_a):
            ib = np.where(bins_a == b)[0]; l[ib] = rng_.permutation(l[ib])
        perms[i] = l
    for j, gname in enumerate(AG):
        y = Cm[:, j].astype(float); obs_d = y[lab].mean() - y[~lab].mean()
        null_d = (perms @ y) / lab.sum() - ((~perms) @ y) / (~lab).sum()
        res.append({"genotype": o.loc[o.index[m[0]], "genotype"], "age": o.loc[o.index[m[0]], "age_mo"], "gene": gname,
                    "n_near_pos": int(lab.sum()), "n_near_neg": int((~lab).sum()),
                    "det_near_pos": 100 * y[lab].mean(), "det_near_neg": 100 * y[~lab].mean(),
                    "p_perm": (np.sum(np.abs(null_d) >= abs(obs_d)) + 1) / (NPERM + 1)})
r7 = pd.DataFrame(res)
r7["q"] = r7.groupby(["genotype", "age"])["p_perm"].transform(lambda p: multipletests(p, method="fdr_bh")[1])
r7.to_csv(TAB / "10xAD_07_astro_near_spp1_mg.csv", index=False)
for (g, a_), t in r7.groupby(["genotype", "age"]):
    print(f"\n== {g} {a_:g} mo (astrocytes next to Spp1+: {t['n_near_pos'].iloc[0]})")
    print(t.round(3).sort_values("p_perm")[["gene", "det_near_pos", "det_near_neg", "p_perm", "q"]].to_string(index=False))

In [ ]:
# ==========================================================
# B4) Identity of Spp1+ microglia: healthy ageing vs amyloid, depth-matched to 100 transcripts
# ==========================================================
from scipy.stats import fisher_exact

MODULES = {
    "Homeostatic":           ["P2ry12", "Tmem119", "Cx3cr1", "Gpr34", "Csf1r", "Hexb", "Olfml3", "P2ry13"],
    "Phagocytic / lysosomal": ["Trem2", "Tyrobp", "Apoe", "Cd63", "Ctsb", "Ctsd", "Ctss", "Cst3", "Npc2", "Grn"],
    "WAM / DAM":             ["Itgax", "Gpnmb", "Igf1", "Lyz2", "Lgals3bp", "Fcgr3", "Cd14"],
    "Antigen presentation":  ["Cd74", "H2-K1", "H2-D1", "B2m"],
    "Interferon":            ["Ifit3", "Irf7", "Cxcl10", "Oasl2", "Ifitm3"]}
MODULES = {k: [g for g in v if g in xa.var_names] for k, v in MODULES.items()}
GL = [g for v in MODULES.values() for g in v]

GROUPS = [("WT", 13.4, 1, "WT 13.4 Spp1+"), ("WT", 13.4, 0, "WT 13.4 Spp1−"),
          ("TgCRND8", 5.7, 1, "Tg 5.7 Spp1+"), ("TgCRND8", 17.9, 1, "Tg 17.9 Spp1+"), ("TgCRND8", 17.9, 0, "Tg 17.9 Spp1−")]

# depth matching: binomial thinning of microglial profiles to T panel transcripts
T = 100
idx_mg = np.where(o["is_mg"].to_numpy())[0]
Xmg = dense(xa[idx_mg].layers["counts"]).astype(np.int64)
dep = Xmg.sum(1); keep = dep >= T
Xmg_t = np.random.default_rng(0).binomial(Xmg[keep], (T / dep[keep])[:, None])
om = o.iloc[idx_mg[keep]].copy()
om["spp1"] = (om["Spp1_n"] >= 1).astype(int)            # Spp1 state defined on raw counts, as before
gidx = {g: list(xa.var_names).index(g) for g in GL}
print(f"microglia with >= {T} transcripts: {keep.sum():,} / {len(keep):,}")

det, ns = {}, {}
for g_, a_, s_, lab in GROUPS:
    sel = ((om["genotype"] == g_) & (om["age_mo"] == a_) & (om["spp1"] == s_)).to_numpy()
    ns[lab] = int(sel.sum())
    det[lab] = {gn: 100 * (Xmg_t[sel, gidx[gn]] >= 1).mean() for gn in GL}
det = pd.DataFrame(det).loc[GL]
print("n cells:", ns)
print(det.round(0).to_string())

# Spp1+ in healthy ageing vs Spp1+ in amyloid (17.9 mo): Fisher per gene, BH across genes
def cnt(lab, gn):
    g_, a_, s_ = [(g, a, s) for g, a, s, l in GROUPS if l == lab][0]
    sel = ((om["genotype"] == g_) & (om["age_mo"] == a_) & (om["spp1"] == s_)).to_numpy()
    d = Xmg_t[sel, gidx[gn]] >= 1
    return int(d.sum()), int((~d).sum())
rows = []
for gn in GL:
    a1, b1 = cnt("WT 13.4 Spp1+", gn); a2, b2 = cnt("Tg 17.9 Spp1+", gn)
    orr, p = fisher_exact([[a2, b2], [a1, b1]])
    mod = [k for k, v in MODULES.items() if gn in v][0]
    rows.append({"module": mod, "gene": gn, "WT_Spp1pos_%": 100 * a1 / (a1 + b1), "Tg_Spp1pos_%": 100 * a2 / (a2 + b2),
                 "OR_Tg_vs_WT": orr, "p": p})
cmp8 = pd.DataFrame(rows); cmp8["q"] = multipletests(cmp8["p"], method="fdr_bh")[1]
cmp8.to_csv(TAB / "10xAD_08_identity_spp1pos_WT_vs_Tg.csv", index=False)
print("\nSpp1+ microglia, amyloid (Tg 17.9) vs healthy ageing (WT 13.4):")
print(cmp8.round(3).to_string(index=False))

# enrichment within genotype: Spp1+ vs Spp1- (difference in percentage points), shows what each state adds
enr = pd.DataFrame({"WT 13.4: Spp1+ minus Spp1−": det["WT 13.4 Spp1+"] - det["WT 13.4 Spp1−"],
                    "Tg 17.9: Spp1+ minus Spp1−": det["Tg 17.9 Spp1+"] - det["Tg 17.9 Spp1−"]})
print("\nenrichment in Spp1+ vs Spp1- within genotype (percentage points):")
print(enr.round(0).to_string())

# Spp1 transcripts per positive microglia (raw counts)
for g_, a_ in [("WT", 13.4), ("TgCRND8", 5.7), ("TgCRND8", 17.9)]:
    v = o.loc[o["is_mg"] & (o["genotype"] == g_) & (o["age_mo"] == a_) & (o["Spp1_n"] >= 1), "Spp1_n"]
    print(f"{g_} {a_:g} mo: Spp1 per positive cell median {v.median():.0f}, mean {v.mean():.1f}, "
          f"share with >= 3 transcripts {100 * (v >= 3).mean():.0f}% (n = {len(v)})")

In [ ]:
# ==========================================================
# B5) Transcript-level astrocyte coverage and reactivity around Spp1+ microglia (all animals)
# ==========================================================
import re
import pyarrow.parquet as pq
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy.spatial import cKDTree
if "C" in globals() and not callable(C): del C

COVER = ["Aldh1l1", "Gja1", "Acsbg1", "Slc39a12", "Ntsr2"]               # homeostatic astrocyte genes = coverage
REACT = ["Gfap", "Serpina3n", "C4b", "Cd44", "Vim", "Osmr", "C3", "Aqp4"]   # reactive / candidate genes
R_IN, R_OUT, QV = 5, 25, 20
o["ctx_bin"] = pd.cut(o["f_oligo"], [-0.01, 0.2, 0.4, 1.0], labels=["Grey", "Border", "Fibre"]).astype(str)


def bundle_of(smp, os_):
    """Bundle folder in data/ for one sample (sample names may or may not be the folder names)."""
    if (DAT / str(smp)).exists(): return str(smp)
    age = float(os_["age_mo"].iloc[0]); wt = str(os_["genotype"].iloc[0]).lower() in ("wt", "wildtype", "wild-type")
    for p in DAT.glob("Xenium_V1_FFPE_*_outs"):
        m = re.search(r"FFPE_(.+?)_(\d+)_(\d+)_months", p.name)
        if m and abs(float(f"{m[2]}.{m[3]}") - age) < 0.01 and ((m[1] == "wildtype") == wt): return p.name
    raise FileNotFoundError(smp)


def load_tx(bundle, genes):
    t = pq.read_table(DAT / bundle / "transcripts.parquet",
                      columns=["cell_id", "feature_name", "x_location", "y_location", "qv"],
                      filters=[("qv", ">=", QV), ("feature_name", "in", [g.encode() for g in genes])]).to_pandas()
    for c in ["cell_id", "feature_name"]:
        if t[c].dtype == object and len(t) and isinstance(t[c].iloc[0], bytes):
            t[c] = t[c].str.decode("utf-8")
    return t


rows, per_mg = [], []
for smp in o["sample"].unique():
    os_ = o[o["sample"] == smp]
    mgs = os_[os_["is_mg"].astype(bool)].copy()
    mgs["spp1"] = (mgs["Spp1_n"] >= 1).astype(int)
    if mgs["spp1"].sum() < 10:
        print(f"{smp}: {mgs['spp1'].sum()} Spp1+ microglia, skipped"); continue
    tx = load_tx(bundle_of(smp, os_), COVER + REACT)
    mg_ids = set(str(i).rsplit("_", 1)[-1] for i in os_.index[os_["cell_type"].astype(str).str.contains("Microglia")])
    tx = tx[~tx["cell_id"].isin(mg_ids)]                                  # drop transcripts assigned to any microglia
    xy_mg = mgs[["x_centroid", "y_centroid"]].to_numpy()
    for g in COVER + REACT:
        tree = cKDTree(tx.loc[tx["feature_name"] == g, ["x_location", "y_location"]].to_numpy())
        mgs[g] = tree.query_ball_point(xy_mg, R_OUT, return_length=True) - tree.query_ball_point(xy_mg, R_IN, return_length=True)
    mgs["cover"] = mgs[COVER].sum(1)
    per_mg.append(mgs.assign(sample=smp))

    geno = os_["genotype"].iloc[0] if "genotype" in os_ else "WT"
    variants = [("all", mgs)]
    if "d_vlmc" in mgs and mgs["d_vlmc"].notna().any():
        variants.append(("no VLMC within 30 µm", mgs[mgs["d_vlmc"] > 30]))
    for var, dm in variants:
        base = {"sample": smp, "genotype": geno, "age": os_["age_mo"].iloc[0], "variant": var,
                "n_pos": int(dm["spp1"].sum()), "n_neg": int((dm["spp1"] == 0).sum())}
        f = smf.glm("cover ~ spp1 + C(ctx_bin)", data=dm, family=sm.families.Poisson()).fit(cov_type="HC1")
        lo, hi = np.exp(f.conf_int().loc["spp1"])
        rows.append({**base, "gene": "coverage (homeostatic astro transcripts)", "ratio": np.exp(f.params["spp1"]),
                     "lo": lo, "hi": hi, "p": f.pvalues["spp1"]})
        for g in REACT:                                      # reactive transcripts per homeostatic astrocyte transcript
            d = dm[(dm[g] + dm["cover"]) > 0]
            X = sm.add_constant(pd.get_dummies(d[["spp1", "ctx_bin"]], columns=["ctx_bin"], drop_first=True).astype(float))
            f = sm.GLM(np.column_stack([d[g], d["cover"]]), X, family=sm.families.Binomial()).fit(cov_type="HC1")
            lo, hi = np.exp(f.conf_int().loc["spp1"])
            rows.append({**base, "gene": g, "ratio": np.exp(f.params["spp1"]), "lo": lo, "hi": hi, "p": f.pvalues["spp1"]})
    del tx

tx_res = pd.DataFrame(rows)
tx_res["q"] = tx_res.groupby(["sample", "variant"])["p"].transform(lambda p: multipletests(p, method="fdr_bh")[1])
tx_res.round(4).to_csv(TAB / "10x_12_transcript_level_astro.csv", index=False)
pd.concat(per_mg).to_csv(TAB / "10x_12_transcript_level_per_microglia.csv")
print(f"transcripts {R_IN} to {R_OUT} µm around each strict microglia, transcripts assigned to microglia removed, qv >= {QV}")
print("coverage: rate ratio Spp1+ vs Spp1- (< 1 = fewer astrocyte transcripts) | genes: odds ratio of gene vs homeostatic astro transcripts")
display(tx_res.pivot_table(index="gene", columns=["genotype", "age", "variant"], values="ratio", sort=False).round(2))
display(tx_res.pivot_table(index="gene", columns=["genotype", "age", "variant"], values="q", sort=False).round(3))
display(tx_res.groupby(["genotype", "age", "variant"])[["n_pos", "n_neg"]].first())

In [ ]:
# ==========================================================
# B6) Figure 1: public cohort, WT 2.5 / 5.7 / 13.4 mo (10x Xenium, coronal)
# ==========================================================
import matplotlib.gridspec as gridspec
from matplotlib.patches import ConnectionPatch
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from scipy.stats import fisher_exact

C_MG, C_POS, C_NEG, C_AS, C_FIB = "#e0a100", "#c2185b", "#f1d48a", "#3b5bdb", "#9fb3c8"
CTX = ["Grey", "Border", "Fibre"]; CTX_LBL = {"Grey": "Grey matter", "Border": "White matter border", "Fibre": "White matter"}; C_CTX = {"Grey": "#b5b5b5", "Border": "#8fa6c8", "Fibre": "#3f5f8f"}
WT_AGES = [2.5, 5.7, 13.4]; A_OLD = 13.4
CTX2_LBL = {"Fibre / border": "White matter\n+ border", "Grey": "Grey matter"}
sig = lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
mw = mg[mg["genotype"] == "WT"].copy(); t_old = mw[mw["age_mo"] == A_OLD]
f_age = smf.glm("spp1 ~ aged + ctx_bin", data=mw, family=sm.families.Binomial()).fit(cov_type="HC1")
print("age OR", np.exp(f_age.params["aged"]), np.exp(f_age.conf_int().loc["aged"]).values, f_age.pvalues["aged"])

def tissue_image(oo, res=40):
    xb = np.arange(oo["x_centroid"].min(), oo["x_centroid"].max() + res, res)
    yb = np.arange(oo["y_centroid"].min(), oo["y_centroid"].max() + res, res)
    H_all, _, _ = np.histogram2d(oo["x_centroid"], oo["y_centroid"], bins=[xb, yb])
    fbm = oo["ctx_bin"].isin(["Fibre", "Border"]).to_numpy()
    H_fb, _, _ = np.histogram2d(oo["x_centroid"][fbm], oo["y_centroid"][fbm], bins=[xb, yb])
    img = np.zeros(H_all.shape + (4,)); img[H_all > 0] = to_rgba("#ececec")
    img[(H_all > 0) & (H_fb / np.maximum(H_all, 1) >= 0.5)] = to_rgba("#bfcde0")
    return img.transpose(1, 0, 2), (xb[0], xb[-1], yb[0], yb[-1])

# astrocyte expression next to Spp1+ vs Spp1- microglia (13.4 mo), depth-normalised, permutation within context
AG10 = [g for g in ["Gfap", "Cd44", "Serpina3n", "Vim", "C4b", "Aqp4", "Aldh1l1"] if g in xa.var_names]
def astro_expr_10x(sample, nperm=5000):
    m = np.where((o["sample"] == sample).to_numpy())[0]; xy = xa.obsm["spatial"][m]
    ia, im = o["is_astro"].to_numpy()[m], o["is_mg"].to_numpy()[m]; ip = im & (o["Spp1_n"].to_numpy()[m] >= 1)
    near_p = cKDTree(xy[ip]).query_ball_point(xy[ia], 25, return_length=True) > 0
    near_n = cKDTree(xy[im & ~ip]).query_ball_point(xy[ia], 25, return_length=True) > 0
    keep = near_p | near_n; lab = near_p[keep]; rows = m[ia][keep]
    strat = o["ctx_bin"].to_numpy()[rows]
    X = dense(xa[rows, AG10].layers["counts"]).astype(float)
    E = np.log1p(X / np.maximum(o["n_counts"].to_numpy()[rows], 1)[:, None] * 100)
    rng2 = np.random.default_rng(0); perms = np.empty((nperm, len(lab)), bool)
    for i in range(nperm):
        l = lab.copy()
        for b in np.unique(strat):
            ib = np.where(strat == b)[0]; l[ib] = rng2.permutation(l[ib])
        perms[i] = l
    out = []
    for j, g in enumerate(AG10):
        e = E[:, j]; d = e[lab].mean() - e[~lab].mean(); nd = (perms @ e) / lab.sum() - ((~perms) @ e) / (~lab).sum()
        out.append({"gene": g, "frac_pos": 100 * (X[lab, j] >= 1).mean(), "frac_neg": 100 * (X[~lab, j] >= 1).mean(),
                    "mean_pos": e[lab].mean(), "mean_neg": e[~lab].mean(),
                    "fold": np.expm1(e[lab].mean()) / max(np.expm1(e[~lab].mean()), 1e-9),
                    "p_expr": (np.sum(np.abs(nd) >= abs(d)) + 1) / (nperm + 1), "n_pos": int(lab.sum()), "n_neg": int((~lab).sum())})
    return pd.DataFrame(out)
s_old = o.loc[(o["genotype"] == "WT") & (o["age_mo"] == A_OLD), "sample"].iloc[0]
ex10 = astro_expr_10x(s_old); print(ex10.round(3).to_string(index=False))
ex10.to_csv(TAB / "Fig2_astro_expression_WT13.csv", index=False)

fig = plt.figure(figsize=(180 * MM, 178 * MM))
gs = gridspec.GridSpec(4, 12, figure=fig, height_ratios=[0.85, 0.16, 0.95, 0.95], hspace=0.55, wspace=2.6,
                       left=0.09, right=0.95, top=0.96, bottom=0.06)

# ================= A: three sections, common scale, connected to a true age axis
secs = [(a_, o[(o["genotype"] == "WT") & (o["age_mo"] == a_)]) for a_ in WT_AGES]
span = max(max(oo["x_centroid"].max() - oo["x_centroid"].min(), oo["y_centroid"].max() - oo["y_centroid"].min())
           for _, oo in secs) * 1.03
axT = fig.add_subplot(gs[1, :])
axT.set_xlim(0, 15); axT.set_ylim(-1, 1); axT.axis("off")
axT.axvspan(10.5, 14.5, ymin=0.25, ymax=0.75, color="#f0f0f0", zorder=0)
axT.plot([0, 15], [0, 0], color="#555555", lw=0.8)
for t_ in range(0, 16, 2):
    axT.plot([t_, t_], [-0.12, 0.12], color="#555555", lw=0.6)
    axT.text(t_, -0.55, str(t_), ha="center", va="center", fontsize=5.5)
axT.text(15.2, 0, "months", va="center", fontsize=5.5)
axT.text(12.5, 0.55, "midlife", ha="center", fontsize=5.2, color="#777777")
for k, (a_, oo) in enumerate(secs):
    ax = fig.add_subplot(gs[0, 4 * k:4 * k + 4])
    col = np.where(oo["ctx_bin"] == "Fibre", "#9fb3c8", np.where(oo["ctx_bin"] == "Border", "#d3dce6", "#efefef"))
    ax.scatter(oo["x_centroid"], oo["y_centroid"], s=0.08, c=col, lw=0, rasterized=True)
    mm_ = oo[oo["is_mg"]]; pp_ = mm_[mm_["Spp1_n"] >= 1]
    ax.scatter(mm_["x_centroid"], mm_["y_centroid"], s=0.4, c=C_MG, lw=0, rasterized=True)
    ax.scatter(pp_["x_centroid"], pp_["y_centroid"], s=7, c=C_POS, lw=0.3, edgecolor="white", zorder=5, rasterized=True)
    cxs = (oo["x_centroid"].min() + oo["x_centroid"].max()) / 2; cys = (oo["y_centroid"].min() + oo["y_centroid"].max()) / 2
    ax.set_xlim(cxs - span / 2, cxs + span / 2); ax.set_ylim(cys + span / 2, cys - span / 2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_anchor("S")                                               # sections sit on the age axis
    ax.plot([cxs + span / 2 - 1100, cxs + span / 2 - 100], [cys + span / 2 - 150] * 2, color="#222222", lw=1.1)
    ax.text(cxs + span / 2 - 600, cys + span / 2 - 400, "1 mm", ha="center", fontsize=5)
    ax.set_title(("# 1A  " if k == 0 else "") + f"WT {a_:g} mo", loc="left", fontsize=7, fontweight="bold" if k == 0 else "normal")
    axT.scatter(a_, 0, s=26, color=C_POS if a_ == A_OLD else "#9aa8dc", zorder=3)
    fig.add_artist(ConnectionPatch(xyA=(0.5, 0.0), coordsA=ax.transAxes, xyB=(a_, 0.12), coordsB=axT.transData,
                                   color="#bbbbbb", lw=0.6, ls=(0, (2, 2))))
    if k == 2:
        for lab_, c_ in [("White matter", "#9fb3c8"), ("White matter border", "#d3dce6"), ("Microglia", C_MG), ("Spp1+ microglia", C_POS)]:
            ax.scatter([], [], s=12, c=c_, label=lab_)
        ax.legend(loc="upper left", bbox_to_anchor=(0.92, 1.0), fontsize=5.3)
        
# ================= B: Spp1+ share over age, one line per context
axB = fig.add_subplot(gs[2, 0:5])
for cx_ in CTX:
    v = [100 * mw[(mw["age_mo"] == a_) & (mw["ctx_bin"] == cx_)]["spp1"].mean() for a_ in WT_AGES]
    axB.plot(WT_AGES, v, "-o", color=C_CTX[cx_], lw=1.4 if cx_ != "Grey" else 1.0, ms=4, label=CTX_LBL[cx_])
axB.axvspan(10.5, 14.5, color="#f3f3f3", zorder=0)
axB.set_xlim(0, 15); axB.set_xticks([0, 2.5, 5.7, 10, 13.4]); axB.set_xticklabels(["0", "2.5", "5.7", "10", "13.4"])
axB.set_xlabel("Age (months)"); axB.set_ylabel("Spp1+ microglia (%)"); axB.set_ylim(0, 12)
axB.legend(title="Local tissue", fontsize=5.5, title_fontsize=5.5, loc="upper left")
axB.text(13.4, 11.2, sig(f_age.pvalues["aged"]), ha="center", fontsize=7)
axB.set_title("# 1B  Spp1+ microglia rise in white matter", loc="left", fontsize=7, fontweight="bold")

# ================= C: identity, lollipop of Spp1+ minus Spp1- (depth-matched), 13.4 months
axC = fig.add_subplot(gs[2, 6:12])
IDG = [g for g in ["P2ry12", "Tmem119", "Cx3cr1", "Hexb", "Cd63", "Ctsb", "Npc2", "Lyz2", "Itgax", "Gpnmb", "Cd74"] if g in det.index]
diff = (det.loc[IDG, "WT 13.4 Spp1+"] - det.loc[IDG, "WT 13.4 Spp1−"])
yy = np.arange(len(IDG))[::-1]
for y_, g in zip(yy, IDG):
    c_ = C_POS if diff[g] > 0 else "#7f8fb3"
    axC.plot([0, diff[g]], [y_, y_], color=c_, lw=1.2); axC.scatter(diff[g], y_, s=26, color=c_, zorder=3)
axC.axvline(0, color="#888888", lw=0.6)
axC.set_yticks(yy); axC.set_yticklabels(IDG, style="italic", fontsize=6)
axC.set_xlabel("Spp1+ minus Spp1− microglia (percentage points detecting)")
lim = max(abs(diff.min()), abs(diff.max())) * 1.15; axC.set_xlim(-lim, lim)
axC.text(-lim * 0.95, yy[0] + 0.9, "lost", fontsize=6, color="#7f8fb3", fontweight="bold")
axC.text(lim * 0.95, yy[0] + 0.9, "gained", fontsize=6, color=C_POS, fontweight="bold", ha="right")
axC.set_ylim(-0.7, len(IDG) + 0.4)
axC.set_title("# 1C  Identity of Spp1+ microglia, WT 13.4 mo", loc="left", fontsize=7, fontweight="bold")

# ================= D: slopegraph, share with astrocyte contact, Spp1- -> Spp1+ (13.4 months)
axD = fig.add_subplot(gs[3, 0:4])
for tissue, ctxs, c_ in [("Fibre / border", ["Fibre", "Border"], "#3f5f8f"), ("Grey", ["Grey"], "#b5b5b5")]:
    Y = t_old[t_old["ctx_bin"].isin(ctxs)]; cont = Y["n_astro25"] >= 1
    v = [100 * cont[Y["spp1"] == 0].mean(), 100 * cont[Y["spp1"] == 1].mean()]
    a = int((cont & (Y["spp1"] == 1)).sum()); b = int((Y["spp1"] == 1).sum()) - a
    c = int((cont & (Y["spp1"] == 0)).sum()); d = int((Y["spp1"] == 0).sum()) - c
    p = fisher_exact([[a, b], [c, d]])[1]
    axD.plot([0, 1], v, "-", color=c_, lw=1.6); axD.scatter([0], [v[0]], s=36, facecolor="white", edgecolor=c_, lw=1.2, zorder=3)
    axD.scatter([1], [v[1]], s=36, color=c_, zorder=3)
    axD.text(1.08, v[1], f"{CTX2_LBL[tissue]}  {sig(p)}", va="center", fontsize=5.8, color=c_)
axD.set_xticks([0, 1]); axD.set_xticklabels(["Spp1−", "Spp1+"]); axD.set_xlim(-0.3, 1.9)
axD.set_ylim(0, 70); axD.set_ylabel("Microglia with\nastrocyte contact (%)")
axD.set_title("# 1D  Astrocyte contact, WT 13.4 mo", loc="left", fontsize=7, fontweight="bold")

# ================= E: transcript-level astrocyte signal around Spp1+ vs Spp1- microglia (13.4 months)
pm12 = pd.read_csv(TAB / "10x_12_transcript_level_per_microglia.csv", index_col=0)
tr12 = pd.read_csv(TAB / "10x_12_transcript_level_astro.csv")
pw = pm12[(pm12["genotype"] == "WT") & (pm12["age_mo"] == A_OLD)].copy()
tw = tr12[(tr12["genotype"] == "WT") & (tr12["age"] == A_OLD) & (tr12["variant"] == "all")].set_index("gene")
pw["ctx2"] = np.where(pw["ctx_bin"].isin(["Border", "Fibre"]), "Fibre / border", "Grey")

def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    b = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return x.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

gsE = gs[3, 5:12].subgridspec(1, 2, wspace=0.9, width_ratios=[1, 1.15])
axE1, axE2 = fig.add_subplot(gsE[0]), fig.add_subplot(gsE[1])
for k, (cx, col) in enumerate([("Fibre / border", "#3d5a8a"), ("Grey", "#a6a6a6")]):
    pts = []
    for j, sp in enumerate([0, 1]):
        m, lo, hi = boot_ci(pw.loc[(pw["ctx2"] == cx) & (pw["spp1"] == sp), "cover"])
        x = j + (k - 0.5) * 0.18
        axE1.plot([x, x], [lo, hi], color=col, lw=0.9)
        axE1.scatter(x, m, s=30, facecolor="white" if sp == 0 else col, edgecolor=col, lw=1.2, zorder=3)
        pts.append((x, m))
    axE1.plot(*zip(*pts), color=col, lw=1.1, zorder=2)
    axE1.text(pts[1][0] + 0.12, pts[1][1], CTX2_LBL[cx], color=col, va="center", fontsize=5.5)
rc = tw.loc["coverage (homeostatic astro transcripts)"]
axE1.text(0.98, 0.98, f"rate ratio {rc['ratio']:.2f}\n[{rc['lo']:.2f}, {rc['hi']:.2f}] {sig(rc['q'])}",
          transform=axE1.transAxes, ha="right", va="top", fontsize=5.3)
axE1.set_xticks([0, 1]); axE1.set_xticklabels(["Spp1−", "Spp1+"]); axE1.set_xlim(-0.4, 2.1); axE1.set_ylim(0, None)
axE1.set_ylabel("Homeostatic astrocyte transcripts\nwithin 5 to 25 µm per microglia")
axE1.set_title("# 1E  Astrocyte transcripts around microglia, WT 13.4 mo", loc="left", fontsize=7, fontweight="bold")

G2E = [g for g in ["Gfap", "Serpina3n", "C4b", "Aqp4", "Cd44", "Vim", "Osmr"] if g in tw.index][::-1]
t = tw.loc[G2E]; yy = np.arange(len(G2E))
axE2.axvline(0, color="#999999", lw=0.6, ls="--")
axE2.hlines(yy, np.log2(t["lo"]), np.log2(t["hi"]), color="#c2185b", lw=1.0)
axE2.scatter(np.log2(t["ratio"]), yy, s=24, color="#c2185b", zorder=3)
for y_, (_, r_) in zip(yy, t.iterrows()):
    axE2.text(np.log2(r_["hi"]) + 0.15, y_, sig(r_["q"]), va="center", fontsize=6 if sig(r_["q"]) != "n.s." else 5)
axE2.set_yticks(yy); axE2.set_yticklabels(G2E, style="italic")
axE2.set_xlim(-1, max(4, np.log2(t["hi"]).max() + 0.9))
axE2.set_xlabel(f"log2 odds ratio, Spp1+ (n = {int(rc['n_pos'])}) vs Spp1− (n = {int(rc['n_neg'])})\n"
                "gene per homeostatic astrocyte transcript")
save(fig, "Fig1")

In [ ]:
# ==========================================================
# B7) Figure 4 (amyloid) with summary schematic 4F and quantification 4G across all cohorts
#     4G reads tables of notebook 01 (Z1, Z2), of this notebook (10xAD_06, 10xAD_07) and of notebook 05 (tables/cosmx C6, C7, C10)
# ==========================================================
import matplotlib.gridspec as gridspec
from matplotlib.patches import ConnectionPatch
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import fisher_exact

C_MG, C_POS = "#e0a100", "#c2185b"
C_FIB, C_BOR, C_BG = "#9fb3c8", "#d3dce6", "#efefef"
C_WT, C_TG = "#3b5bdb", "#7a3e9d"
OLD = {"WT": 13.4, "TgCRND8": 17.9}
AGES = {"WT": [2.5, 5.7, 13.4], "TgCRND8": [2.5, 5.7, 17.9]}
GC = {"WT": C_WT, "TgCRND8": C_TG}; GN = {"WT": "WT", "TgCRND8": "TgCRND8"}
sig = lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
CM_DOT = LinearSegmentedColormap.from_list("dot", ["#f1f3fb", "#9aa8dc", "#2b3f9e"])

# ---- astrocyte expression next to Spp1+ vs Spp1- microglia (oldest animal per genotype)
AGF = [g for g in ["Gfap", "Serpina3n", "Vim", "Osmr", "Cd44", "C4b", "Aldh1l1"] if g in xa.var_names]
def astro_expr_10x(sample, genes, nperm=5000):
    m = np.where((o["sample"] == sample).to_numpy())[0]; xy = xa.obsm["spatial"][m]
    ia, im = o["is_astro"].to_numpy()[m], o["is_mg"].to_numpy()[m]; ip = im & (o["Spp1_n"].to_numpy()[m] >= 1)
    near_p = cKDTree(xy[ip]).query_ball_point(xy[ia], 25, return_length=True) > 0
    near_n = cKDTree(xy[im & ~ip]).query_ball_point(xy[ia], 25, return_length=True) > 0
    keep = near_p | near_n; lab = near_p[keep]; rows = m[ia][keep]; strat = o["ctx_bin"].to_numpy()[rows]
    X = dense(xa[rows, genes].layers["counts"]).astype(float)
    E = np.log1p(X / np.maximum(o["n_counts"].to_numpy()[rows], 1)[:, None] * 100)
    rng2 = np.random.default_rng(0); perms = np.empty((nperm, len(lab)), bool)
    for i in range(nperm):
        l = lab.copy()
        for b in np.unique(strat):
            ib = np.where(strat == b)[0]; l[ib] = rng2.permutation(l[ib])
        perms[i] = l
    out = []
    for j, g in enumerate(genes):
        e = E[:, j]; d = e[lab].mean() - e[~lab].mean(); nd = (perms @ e) / lab.sum() - ((~perms) @ e) / (~lab).sum()
        out.append({"gene": g, "mean_pos": e[lab].mean(), "mean_neg": e[~lab].mean(),
                    "log2fc": np.log2((np.expm1(e[lab].mean()) + 0.01) / (np.expm1(e[~lab].mean()) + 0.01)),
                    "p_expr": (np.sum(np.abs(nd) >= abs(d)) + 1) / (nperm + 1), "n_pos": int(lab.sum())})
    return pd.DataFrame(out)
exF = {g: astro_expr_10x(o.loc[(o["genotype"] == g) & (o["age_mo"] == a_), "sample"].iloc[0], AGF) for g, a_ in OLD.items()}
for g, d in exF.items(): print(g); print(d.round(3).to_string(index=False))

# ---- microglial identity: detection + mean expression after thinning to 100 transcripts (from cell 8)
ID_SEL = {"Homeostatic": ["P2ry12", "Tmem119", "P2ry13", "Csf1r", "Hexb"],
          "Lysosomal": ["Trem2", "Tyrobp", "Cd63", "Ctsb", "Npc2"],
          "DAM / WAM": ["Itgax", "Gpnmb", "Igf1", "Lyz2"], "MHC": ["Cd74", "H2-K1", "H2-D1"]}
ID_SEL = {k: [g for g in v if g in gidx] for k, v in ID_SEL.items()}
gl = [g for v in ID_SEL.values() for g in v]
ID_ROWS = ["WT 13.4 Spp1−", "WT 13.4 Spp1+", "Tg 5.7 Spp1+", "Tg 17.9 Spp1−", "Tg 17.9 Spp1+"]
idm, idf = [], []
for lab in ID_ROWS:
    g_, a_, s_ = [(g, a, s) for g, a, s, l in GROUPS if l == lab][0]
    sel = ((om["genotype"] == g_) & (om["age_mo"] == a_) & (om["spp1"] == s_)).to_numpy()
    V = Xmg_t[sel][:, [gidx[g] for g in gl]]
    idm.append(np.log1p(V).mean(0)); idf.append(100 * (V >= 1).mean(0))
idm, idf = np.array(idm), np.array(idf)

fig = plt.figure(figsize=(180 * MM, 250 * MM))
GO = gridspec.GridSpec(5, 1, figure=fig, height_ratios=[0.78, 0.95, 0.95, 0.62, 0.95], hspace=0.62,
                       left=0.10, right=0.95, top=0.965, bottom=0.04)
row = lambda i, **kw: GO[i].subgridspec(1, 12, wspace=2.6, **kw)
gTop = GO[0].subgridspec(2, 12, height_ratios=[1, 0.22], hspace=0.04, wspace=2.6)
gBC, gDE = row(1), row(2)

# ================= A: sections + two-track age axis
secs = {g: o[(o["genotype"] == g) & (o["age_mo"] == a_)] for g, a_ in OLD.items()}
span = max(max(oo["x_centroid"].max() - oo["x_centroid"].min(), oo["y_centroid"].max() - oo["y_centroid"].min())
           for oo in secs.values()) * 1.03
axT = fig.add_subplot(gTop[1, :]); axT.set_xlim(0, 19); axT.set_ylim(-1, 1); axT.axis("off")
axT.axvspan(10.5, 14.5, color="#f0f0f0", zorder=0)
for g, y_ in [("WT", 0.4), ("TgCRND8", -0.4)]:
    axT.plot([0, 19], [y_, y_], color="#cccccc", lw=0.6)
    axT.scatter(AGES[g], [y_] * 3, s=20, color=[GC[g] if a == OLD[g] else "#cccccc" for a in AGES[g]], zorder=3)
    axT.text(-0.3, y_, "WT" if g == "WT" else "TgCRND8", ha="right", va="center", fontsize=5.8, color=GC[g])
for t_ in range(0, 20, 2):
    axT.text(t_, -0.95, str(t_), ha="center", va="center", fontsize=5.3)
axT.text(19.3, -0.95, "months", va="center", fontsize=5.3)
for k, (g, oo) in enumerate(secs.items()):
    ax = fig.add_subplot(gTop[0, 6 * k:6 * k + 6])
    col = np.where(oo["ctx_bin"] == "Fibre", C_FIB, np.where(oo["ctx_bin"] == "Border", C_BOR, C_BG))
    ax.scatter(oo["x_centroid"], oo["y_centroid"], s=0.1, c=col, lw=0, rasterized=True)
    mm_ = oo[oo["is_mg"]]; pp_ = mm_[mm_["Spp1_n"] >= 1]
    ax.scatter(mm_["x_centroid"], mm_["y_centroid"], s=0.5, c=C_MG, lw=0, rasterized=True)
    ax.scatter(pp_["x_centroid"], pp_["y_centroid"], s=7, c=C_POS, lw=0.3, edgecolor="white", zorder=5, rasterized=True)
    cxs = (oo["x_centroid"].min() + oo["x_centroid"].max()) / 2; cys = (oo["y_centroid"].min() + oo["y_centroid"].max()) / 2
    ax.set_xlim(cxs - span / 2, cxs + span / 2); ax.set_ylim(cys + span / 2, cys - span / 2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_anchor("S")                                               # sections sit on the age axis
    ax.plot([cxs + span / 2 - 1100, cxs + span / 2 - 100], [cys + span / 2 - 150] * 2, color="#222222", lw=1.1)
    ax.text(cxs + span / 2 - 600, cys + span / 2 - 400, "1 mm", ha="center", fontsize=5)
    ax.set_title(("# 4A  " if k == 0 else "") + f"{GN[g]} {OLD[g]:g} mo", loc="left", fontsize=7,
                 fontweight="bold" if k == 0 else "normal", color="#222222")
    fig.add_artist(ConnectionPatch(xyA=(0.5, 0.0), coordsA=ax.transAxes, xyB=(OLD[g], 0.4 if g == "WT" else -0.4),
                                   coordsB=axT.transData, color="#bbbbbb", lw=0.6, ls=(0, (2, 2))))
    if k == 1:
        for lab_, c_ in [("White matter", C_FIB), ("White matter border", C_BOR), ("Microglia", C_MG), ("Spp1+ microglia", C_POS)]:
            ax.scatter([], [], s=12, c=c_, label=lab_)
        ax.legend(loc="upper left", bbox_to_anchor=(0.88, 1.0), fontsize=5.3)

# ================= B: Spp1+ over age, fibre/border (solid) vs grey (dashed), both genotypes
axB = fig.add_subplot(gBC[0, 0:5])
for g in ["WT", "TgCRND8"]:
    for tissue, ctxs, ls in [("white matter", ["Fibre", "Border"], "-"), ("grey matter", ["Grey"], "--")]:
        v = [100 * mg[(mg["genotype"] == g) & (mg["age_mo"] == a_) & mg["ctx_bin"].isin(ctxs)]["spp1"].mean() for a_ in AGES[g]]
        axB.plot(AGES[g], v, ls, color=GC[g], lw=1.4, marker="o", ms=3.5,
                 mfc=GC[g] if ls == "-" else "white", label=f"{'WT' if g == 'WT' else 'TgCRND8'}, {tissue}")
axB.axvspan(10.5, 14.5, color="#f3f3f3", zorder=0)
axB.set_xlim(0, 19); axB.set_xticks([0, 2.5, 5.7, 13.4, 17.9]); axB.set_xticklabels(["0", "2.5", "5.7", "13.4", "17.9"], fontsize=5.8)
axB.set_xlabel("Age (months)"); axB.set_ylabel("Spp1+ microglia (%)"); axB.set_ylim(0, 12)
axB.legend(fontsize=5.2, loc="upper left")
axB.set_title("# 4B  Where Spp1+ microglia arise", loc="left", fontsize=7, fontweight="bold")

# ================= C: astrocyte signal around Spp1- and Spp1+ microglia, WT 13.4 mo vs TgCRND8 17.9 mo (transcript level)
pm12 = pd.read_csv(TAB / "10x_12_transcript_level_per_microglia.csv", index_col=0)
pm12["tg"] = (pm12["genotype"] != "WT").astype(int)
dC = pm12[((pm12["tg"] == 0) & (pm12["age_mo"] == 13.4)) | ((pm12["tg"] == 1) & (pm12["age_mo"] == 17.9))].copy()
dC["gfap_share"] = dC["Gfap"] / (dC["Gfap"] + dC["cover"]).replace(0, np.nan)

def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; b = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return x.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

# interaction Spp1 state x genotype (section and context as fixed effects)
f_cov = smf.glm("cover ~ spp1 + spp1:tg + C(sample):C(ctx_bin)", data=dC, family=sm.families.Poisson()).fit(cov_type="HC1")
dd = dC[(dC["Gfap"] + dC["cover"]) > 0]
X = pd.get_dummies(dd["sample"].astype(str) + "|" + dd["ctx_bin"].astype(str), drop_first=True).astype(float)
X["spp1"] = dd["spp1"]; X["spp1_tg"] = dd["spp1"] * dd["tg"]
f_gf = sm.GLM(np.column_stack([dd["Gfap"], dd["cover"]]), sm.add_constant(X), family=sm.families.Binomial()).fit(cov_type="HC1")
P_INT = {"cover": f_cov.pvalues["spp1:tg"], "gfap_share": f_gf.pvalues["spp1_tg"]}
print("interaction p: coverage", P_INT["cover"], "| Gfap share", P_INT["gfap_share"])

gsC = gBC[0, 6:12].subgridspec(1, 2, wspace=0.9)
axC1, axC2 = fig.add_subplot(gsC[0]), fig.add_subplot(gsC[1])
for ax_, var, ylab, scale in [(axC1, "cover", "Homeostatic astrocyte transcripts\nwithin 5 to 25 µm per microglia", 1),
                              (axC2, "gfap_share", "Gfap share of astrocyte\ntranscripts (%)", 100)]:
    for k, (tg, col, lab) in enumerate([(0, C_WT, "WT 13.4 mo"), (1, C_TG, "TgCRND8 17.9 mo")]):
        pts = []
        for j, sp in enumerate([0, 1]):
            m, lo, hi = boot_ci(dC.loc[(dC["tg"] == tg) & (dC["spp1"] == sp), var])
            x = j + (k - 0.5) * 0.14
            ax_.plot([x, x], [scale * lo, scale * hi], color=col, lw=0.9)
            ax_.scatter(x, scale * m, s=28, facecolor="white" if sp == 0 else col, edgecolor=col, lw=1.2, zorder=3)
            pts.append((x, scale * m))
        ax_.plot(*zip(*pts), color=col, lw=1.1, zorder=2)
        if ax_ is axC1: ax_.text(pts[1][0] + 0.17, pts[1][1], lab, color=col, va="center", fontsize=5.5)
    ax_.text(0.5, 1.0, f"Spp1 x genotype {sig(P_INT[var])}", transform=ax_.transAxes, ha="center", va="bottom", fontsize=5.5)
    ax_.set_xticks([0, 1]); ax_.set_xticklabels(["Spp1−", "Spp1+"]); ax_.set_xlim(-0.4, 1.9 if ax_ is axC1 else 1.4)
    ax_.set_ylim(0, None); ax_.set_ylabel(ylab)
axC1.set_title("# 4C  Astrocyte transcripts around microglia", loc="left", fontsize=7, fontweight="bold", pad=10)

# ================= 3E: microglial identity dot plot (size = % detecting, colour = mean expression scaled per gene)
axE = fig.add_subplot(gDE[0, 5:12])
sc_ = (idm - idm.min(0)) / np.maximum(idm.max(0) - idm.min(0), 1e-9)
for i in range(len(ID_ROWS)):
    for j in range(len(gl)):
        axE.scatter(j, i, s=4 + 1.1 * idf[i, j], c=[CM_DOT(sc_[i, j])], edgecolor="#555555", lw=0.3, zorder=3)
x0 = 0
for kk, v in ID_SEL.items():
    axE.text(x0 + (len(v) - 1) / 2, -0.95, kk, ha="center", fontsize=5.6)
    if x0 > 0: axE.axvline(x0 - 0.5, color="#dddddd", lw=0.6)
    x0 += len(v)
axE.axhline(1.5, color="#dddddd", lw=0.6)
axE.set_yticks(range(len(ID_ROWS))); axE.set_yticklabels([" ".join(r.replace("Tg", "TgCRND8").split()[:2]) + " mo · " + r.split()[-1] for r in ID_ROWS], fontsize=5.6)
axE.set_xticks(range(len(gl))); axE.set_xticklabels(gl, rotation=60, ha="right", style="italic", fontsize=5.6)
axE.set_xlim(-0.6, len(gl) - 0.4); axE.set_ylim(len(ID_ROWS) - 0.5, -1.3)
for s_ in axE.spines.values(): s_.set_visible(False)
axE.tick_params(length=0)
for v in [25, 50, 75]:
    axE.scatter([], [], s=4 + 1.1 * v, c="#cccccc", edgecolor="#555555", lw=0.3, label=f"{v}%")
axE.legend(fontsize=4.8, loc="upper left", bbox_to_anchor=(1.0, 1.0), labelspacing=1.0, handletextpad=0.2)
cax = axE.inset_axes([1.07, 0.0, 0.025, 0.35])
cb = plt.colorbar(plt.cm.ScalarMappable(cmap=CM_DOT), cax=cax); cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"])
cb.ax.tick_params(labelsize=4.8); cb.outline.set_visible(False)
axE.set_title("# 4E  Microglial identity (depth-matched)", loc="left", fontsize=7, fontweight="bold", pad=14)

# ================= 3D: astrocytes adjacent to Spp1+ vs Spp1- microglia, log2 fold change of expression
axF = fig.add_subplot(gDE[0, 0:3])
yy = np.arange(len(AGF))[::-1]
for k, g in enumerate(["WT", "TgCRND8"]):
    d = exF[g].set_index("gene").loc[AGF]
    nd = ((d["mean_pos"] == 0) | (d["mean_neg"] == 0)).to_numpy()     # gene absent in one group: fold change undefined
    vals = np.where(nd, 0, d["log2fc"]); off = 0.2 if k == 0 else -0.2
    axF.barh(yy + off, vals, height=0.38, color=GC[g],
             label=f"{'WT 13.4 mo' if g == 'WT' else 'TgCRND8 17.9 mo'}, n = {int(d['n_pos'].iloc[0])}")
    for y_, v_, p_, n_ in zip(yy, vals, d["p_expr"], nd):
        if n_:
            axF.text(0.06, y_ + off, "n.d.", va="center", fontsize=5, color=GC[g]); continue
        lab = sig(p_)
        if lab != "n.s.":
            axF.text(v_ + (0.08 if v_ >= 0 else -0.08), y_ + (0.2 if k == 0 else -0.2), lab, va="center",
                     ha="left" if v_ >= 0 else "right", fontsize=6)
axF.axvline(0, color="#999999", lw=0.6)
axF.set_yticks(yy); axF.set_yticklabels(AGF, style="italic", fontsize=6)
axF.set_xlabel("log2 fold change, astrocytes adjacent to\nSpp1+ vs Spp1− microglia")
axF.legend(fontsize=4.8, loc="lower right", bbox_to_anchor=(1.02, -0.02), ncol=1, handlelength=0.8, handletextpad=0.3)
axF.set_xlim(-0.5, np.nanmax([exF[g]["log2fc"].max() for g in exF]) + 0.6)
axF.set_title("# 4D  Reactive neighbouring astrocytes", loc="left", fontsize=7, fontweight="bold")

# ================= F: summary schematic, astrocyte context of Spp1+ microglia in midlife, old age and amyloid pathology
from matplotlib.patches import Circle, Rectangle, RegularPolygon, FancyBboxPatch
gF = GO[3].subgridspec(1, 3, wspace=0.08)
C_Q, C_R, C_RR = "#cdd6f2", "#6f86e0", "#24399e"           # quiet, reactive, strongly reactive astrocyte
C_GM, C_WMs = "#f2f2f2", "#d6dfeb"
rs_ = np.random.default_rng(7)
gx, gy = np.meshgrid(np.linspace(0.13, 1.87, 11), np.linspace(0.13, 0.87, 5))
PTS = np.column_stack([gx.ravel(), gy.ravel()]) + rs_.uniform(-0.035, 0.035, (gx.size, 2))   # same astrocyte layout in all panels

def astro(ax, x, y, c, r=0.045):
    ax.add_patch(RegularPolygon((x, y), numVertices=6, radius=r, orientation=0.3, fc=c, ec="white", lw=0.4, zorder=3))
    for a_ in np.linspace(0, 2 * np.pi, 6, endpoint=False) + 0.3:
        ax.plot([x, x + 1.9 * r * np.cos(a_)], [y, y + 1.9 * r * np.sin(a_)], color=c, lw=0.7, zorder=2)

def micro(ax, x, y):
    ax.add_patch(Circle((x, y), 0.045, fc=C_POS, ec="white", lw=0.5, zorder=5))

def panel(ax, title, sub, mode):
    ax.set_xlim(0, 2); ax.set_ylim(0, 1.15); ax.set_aspect("equal"); ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 1.96, 0.96, boxstyle="round,pad=0,rounding_size=0.05", fc=C_GM, ec="none", zorder=0))
    ax.add_patch(Rectangle((0.02, 0.38), 1.96, 0.3, fc=C_WMs, ec="none", zorder=0))     # white matter band
    pts = PTS
    cl = np.array([[0.95, 0.55], [1.08, 0.5], [1.0, 0.44]])                              # Spp1+ cluster in white matter
    if mode == "amyloid":
        ax.add_patch(Circle((0.45, 0.8), 0.08, fc="#bdbdbd", ec="#9e9e9e", lw=0.5, hatch="////", zorder=4))
        cl = np.vstack([[[0.33, 0.72], [0.57, 0.73], [0.45, 0.94]], cl[:2] + [0.25, 0]])
    d = np.min(np.linalg.norm(pts[:, None, :] - cl[None, :, :], axis=2), axis=1)
    for (x, y), d_ in zip(pts, d):
        if mode == "midlife":
            if d_ < 0.2: continue                                                        # astrocyte-poor spot
            c = C_RR if d_ < 0.32 else C_Q
        elif mode == "old":
            c = C_RR if d_ < 0.25 else C_R
        else:
            c = C_R if d_ > 0.25 else C_RR if rs_.random() < 0.3 else C_R
        if mode == "amyloid" and np.hypot(x - 0.45, y - 0.8) < 0.12: continue
        astro(ax, x, y, c)
    if mode == "midlife":
        ax.add_patch(Circle((1.01, 0.5), 0.2, fill=False, ec="#888888", lw=0.5, ls=(0, (2, 2)), zorder=4))
    for x, y in cl: micro(ax, x, y)
    ax.text(0.02, 1.07, title, fontsize=6.5, fontweight="bold", va="center")
    ax.text(1.0, -0.09, sub, fontsize=5.6, ha="center", va="top", color="#333333")

titles = [("Midlife (12 to 13.4 mo)", "few astrocytes nearby\nlocal reactivity", "midlife"),
          ("Old age (24 to 25 mo)", "as many astrocytes as elsewhere\nwidespread and local reactivity", "old"),
          ("Amyloid (TgCRND8)", "as many astrocytes as elsewhere\nwidespread reactivity", "amyloid")]
axSs = []
for k, (t_, s_, m_) in enumerate(titles):
    axS = fig.add_subplot(gF[k]); panel(axS, t_, s_, m_); axSs.append(axS)
    if k == 0: axS.set_title("# 4F  Astrocyte context of Spp1+ microglia", loc="left", fontsize=7, fontweight="bold", pad=12)
h_ = [plt.Line2D([], [], marker="o", ls="", ms=5, mfc=C_POS, mec="white", label="Spp1+ microglia"),
      plt.Line2D([], [], marker="h", ls="", ms=6, mfc=C_Q, mec="white", label="quiet astrocyte"),
      plt.Line2D([], [], marker="h", ls="", ms=6, mfc=C_R, mec="white", label="reactive astrocyte"),
      plt.Line2D([], [], marker="h", ls="", ms=6, mfc=C_RR, mec="white", label="strongly reactive astrocyte"),
      Rectangle((0, 0), 1, 1, fc=C_WMs, label="white matter"),
      Rectangle((0, 0), 1, 1, fc="#bdbdbd", hatch="////", ec="#9e9e9e", label="amyloid plaque")]
axSs[1].legend(handles=h_, loc="upper center", bbox_to_anchor=(0.5, -0.27), ncol=6, fontsize=5.4, frameon=False, handletextpad=0.3, columnspacing=1.2)

# ================= G: quantification behind the schematic, same measures in all cohorts
from scipy.stats import ttest_rel
TABC = TAB / "cosmx"
C_MID, C_AGE = "#6f86e0", "#24399e"                                    # midlife, old age (amyloid = C_TG)
f_or = lambda d: smf.glm("spp1 ~ n_astro25 + C(ctx_bin)", data=d, family=sm.families.Binomial()).fit(cov_type="HC1")

# (i) astrocyte neighbours: OR per astrocyte within 25 µm, Spp1+ vs Spp1−, tissue context as fixed effect
z1 = pd.read_csv(TAB / "Z1_age_x_astrocyte_interaction.csv", index_col=0)
z1r = z1[(z1.index == "all three sections | n_astro25") & (z1["model"] == "slope Healthy 1 y")].iloc[0]
m10 = pd.read_csv(TAB / "10xAD_06_strict_microglia.csv", index_col=0)
def or10(g, a_):
    f = f_or(m10[(m10["genotype"] == g) & (m10["age_mo"] == a_)]); lo, hi = np.exp(f.conf_int().loc["n_astro25"])
    return np.exp(f.params["n_astro25"]), lo, hi, f.pvalues["n_astro25"]
c6 = pd.read_csv(TABC / "C6_astro_neighbours_per_mouse.csv", dtype={"mouse": str}).query("group == 'aged'").set_index("mouse")
lb, se_ = np.log(c6["OR"]), (np.log(c6["hi"]) - np.log(c6["lo"])) / 3.92        # fixed-effect meta-analysis of the 4 aged mice
w_ = 1 / se_ ** 2; lp = (w_ * lb).sum() / w_.sum(); sp_ = np.sqrt(1 / w_.sum())
from scipy.stats import norm
OR_POOL = (np.exp(lp), np.exp(lp - 1.96 * sp_), np.exp(lp + 1.96 * sp_), 2 * norm.sf(abs(lp / sp_)))

# (ii) local reactivity: Gfap log2 fold change, astrocytes adjacent to Spp1+ vs Spp1− microglia
z2 = pd.read_csv(TAB / "Z2_astro_near_spp1_shell_radius_controls.csv")
z2r = z2[(z2["sample"] == "Healthy 1 y") & (z2["config"] == "0-25 µm") & (z2["gene"] == "Gfap")].iloc[0]
c7 = pd.read_csv(TABC / "C7_astro_genes_per_mouse.csv", dtype={"mouse": str}).query("group == 'aged' and gene == 'Gfap'").set_index("mouse")
gf10 = {g: exF[g].set_index("gene").loc["Gfap"] for g in exF}

# (iii) widespread reactivity: Gfap+ astrocytes not adjacent to Spp1+ microglia, young reference vs stage
a7 = pd.read_csv(TAB / "10xAD_07_astro_near_spp1_mg.csv").query("gene == 'Gfap'")
def far10(g, a_):
    r = a7[(a7["genotype"] == g) & (a7["age"] == a_)].iloc[0]; return r["det_near_neg"], int(r["n_near_neg"])
def fisher_pct(p1, n1, p2, n2):
    k1, k2 = round(p1 * n1 / 100), round(p2 * n2 / 100); return fisher_exact([[k1, n1 - k1], [k2, n2 - k2]])[1]
c10 = pd.read_csv(TABC / "C10_widespread_vs_local.csv", dtype={"mouse": str}).query("gene == 'Gfap'").set_index("mouse")["det_far_wm"]
PAIR = {"236": "YF01", "237": "YF02", "239": "YF03", "243": "YF04"}            # young and aged section on the same slide
P_WIDE = ttest_rel([c10[a] for a in PAIR], [c10[y] for y in PAIR.values()]).pvalue

rows4 = []
r_ = or10("WT", 13.4)
rows4.append(dict(label="WT 12 mo (own)", stage="Midlife", c=C_MID, OR=z1r["OR"], OR_lo=z1r["lo"], OR_hi=z1r["hi"], OR_p=z1r["p"],
                  l2fc=np.log2(z2r["fold_expr"]), l2fc_p=z2r["p_expr"], far_young=np.nan, far_stage=np.nan, far_p=np.nan))
wy, wo = far10("WT", 2.5), far10("WT", 13.4)
rows4.append(dict(label="WT 13.4 mo (10x)", stage="Midlife", c=C_MID, OR=r_[0], OR_lo=r_[1], OR_hi=r_[2], OR_p=r_[3],
                  l2fc=gf10["WT"]["log2fc"], l2fc_p=gf10["WT"]["p_expr"], far_young=wy[0], far_stage=wo[0], far_p=fisher_pct(*wo, *wy)))
for m_ in PAIR:
    rows4.append(dict(label=f"WT {25 if m_ == '243' else 24} mo, #{m_}", stage="Old age", c=C_AGE,
                      OR=c6.loc[m_, "OR"], OR_lo=c6.loc[m_, "lo"], OR_hi=c6.loc[m_, "hi"], OR_p=c6.loc[m_, "p"],
                      l2fc=c7.loc[m_, "log2FC"], l2fc_p=c7.loc[m_, "p"], far_young=c10[PAIR[m_]], far_stage=c10[m_], far_p=np.nan))
rows4.append(dict(label="Pooled, n = 4", stage="Old age", c=C_AGE, OR=OR_POOL[0], OR_lo=OR_POOL[1], OR_hi=OR_POOL[2], OR_p=OR_POOL[3],
                  l2fc=c7["log2FC"].mean(), l2fc_p=np.nan, far_young=np.mean([c10[y] for y in PAIR.values()]),
                  far_stage=np.mean([c10[a] for a in PAIR]), far_p=P_WIDE, pooled=True))
r_ = or10("TgCRND8", 17.9); ty, to = far10("TgCRND8", 2.5), far10("TgCRND8", 17.9)
rows4.append(dict(label="TgCRND8 17.9 mo (10x)", stage="Amyloid", c=C_TG, OR=r_[0], OR_lo=r_[1], OR_hi=r_[2], OR_p=r_[3],
                  l2fc=gf10["TgCRND8"]["log2fc"], l2fc_p=gf10["TgCRND8"]["p_expr"], far_young=ty[0], far_stage=to[0], far_p=fisher_pct(*to, *ty)))
G4TAB = pd.DataFrame(rows4).drop(columns="c")
G4TAB["pooled"] = G4TAB.get("pooled", False).fillna(False).astype(bool)
print(G4TAB.round(3).to_string(index=False))

gG = GO[4].subgridspec(1, 4, width_ratios=[1, 1, 1, 0.22], wspace=0.12)
axG = [fig.add_subplot(gG[i]) for i in range(3)]; axL = fig.add_subplot(gG[3])
yy4 = np.arange(len(G4TAB))
for ax_ in axG + [axL]:
    ax_.set_ylim(len(G4TAB) - 0.4, -0.6)
    for s_ in ["Midlife", "Old age", "Amyloid"]:
        ii = np.where(G4TAB["stage"] == s_)[0]
        if s_ == "Old age": ax_.axhspan(ii.min() - 0.5, ii.max() + 0.5, color="#f1f3fb", zorder=0, lw=0)
for ax_ in axG[1:]: ax_.set_yticks([])
axG[0].set_yticks(yy4); axG[0].set_yticklabels(G4TAB["label"], fontsize=5.6)
for t_, r in zip(axG[0].get_yticklabels(), rows4): t_.set_color(r["c"]); t_.set_fontweight("bold" if r.get("pooled") else "normal")
for ax_ in axG: ax_.tick_params(axis="y", length=0)

# (i) forest plot of astrocyte neighbour ORs
a_ = axG[0]; a_.axvline(1, color="#999999", lw=0.6, ls=(0, (2, 2)))
for y_, r in zip(yy4, rows4):
    a_.plot([r["OR_lo"], r["OR_hi"]], [y_, y_], color=r["c"], lw=0.9)
    a_.scatter(r["OR"], y_, s=26 if r.get("pooled") else 16, marker="D" if r.get("pooled") else "o", color=r["c"], zorder=3)
    lab = sig(r["OR_p"]); a_.text(r["OR_hi"] * 1.08, y_, lab, va="center", fontsize=5.3, color="#333333" if lab != "n.s." else "#999999")
a_.set_xscale("log"); a_.set_xlim(0.15, 3); a_.set_xticks([0.25, 0.5, 1, 2]); a_.set_xticklabels(["0.25", "0.5", "1", "2"], fontsize=5.6)
a_.minorticks_off(); a_.set_xlabel("OR per astrocyte within 25 µm\n(fewer  ←  →  more)", fontsize=6)
a_.text(0.5, 1.0, "Astrocyte neighbours", transform=a_.transAxes, ha="center", va="bottom", fontsize=6.3)

# (ii) local Gfap
a_ = axG[1]; a_.axvline(0, color="#999999", lw=0.6)
for y_, r in zip(yy4, rows4):
    a_.plot([0, r["l2fc"]], [y_, y_], color=r["c"], lw=0.9)
    a_.scatter(r["l2fc"], y_, s=26 if r.get("pooled") else 16, marker="D" if r.get("pooled") else "o", color=r["c"], zorder=3)
    if not np.isnan(r["l2fc_p"]):
        lab = sig(r["l2fc_p"]); a_.text(r["l2fc"] + 0.12, y_, lab, va="center", fontsize=5.3, color="#333333" if lab != "n.s." else "#999999")
a_.set_xlim(-0.3, 3.6); a_.set_xlabel("Gfap log2 fold change, astrocytes\nadjacent to Spp1+ vs Spp1− microglia", fontsize=6)
a_.text(0.5, 1.0, "Local reactivity", transform=a_.transAxes, ha="center", va="bottom", fontsize=6.3)

# (iii) widespread Gfap: young reference (open) to stage (filled)
a_ = axG[2]
for y_, r in zip(yy4, rows4):
    if np.isnan(r["far_young"]):
        a_.text(50, y_, "n.a.", va="center", ha="center", fontsize=5.3, color="#999999"); continue
    a_.plot([r["far_young"], r["far_stage"]], [y_, y_], color=r["c"], lw=0.9)
    a_.scatter(r["far_young"], y_, s=16, facecolor="white", edgecolor=r["c"], lw=0.9, zorder=3)
    a_.scatter(r["far_stage"], y_, s=26 if r.get("pooled") else 16, marker="D" if r.get("pooled") else "o", color=r["c"], zorder=3)
    if not np.isnan(r["far_p"]):
        lab = sig(r["far_p"]); a_.text(max(r["far_young"], r["far_stage"]) + 2.5, y_, lab, va="center", fontsize=5.3,
                                       color="#333333" if lab != "n.s." else "#999999")
a_.set_xlim(30, 100); a_.set_xlabel("Gfap+ astrocytes not adjacent\nto Spp1+ microglia (%)", fontsize=6)
a_.scatter([], [], s=14, facecolor="white", edgecolor="#555555", label="young reference"); a_.scatter([], [], s=14, color="#555555", label="stage")
a_.legend(fontsize=4.8, loc="upper right", handletextpad=0.2, borderpad=0.2)
a_.text(0.5, 1.0, "Widespread reactivity", transform=a_.transAxes, ha="center", va="bottom", fontsize=6.3)

# stage labels on the right
axL.axis("off"); axL.set_xlim(0, 1)
for s_, sub_, c_ in [("Midlife", "Xenium", C_MID), ("Old age", "CosMx", C_AGE), ("Amyloid", "Xenium", C_TG)]:
    ii = np.where(G4TAB["stage"] == s_)[0]
    axL.plot([0.05, 0.05], [ii.min() - 0.3, ii.max() + 0.3], color=c_, lw=1.2)
    axL.text(0.18, (ii.min() + ii.max()) / 2, f"{s_}\n{sub_}", va="center", fontsize=5.6, color=c_, fontweight="bold", linespacing=1.1)
tG = axG[0].set_title("# 4G  Quantification across cohorts", loc="left", fontsize=7, fontweight="bold", pad=17)
tG.set_x(-0.62)
save(fig, "Fig4")

In [ ]:
# ==========================================================
# B8) Supplementary Tables S1 and S5: source data of Figures 1 and 4
# ==========================================================
from openpyxl.styles import Font
from scipy.stats import fisher_exact
import statsmodels.api as sm, statsmodels.formula.api as smf
XLS = OUT / "final_data" / "supplementary_tables"; XLS.mkdir(parents=True, exist_ok=True)
if "C" in globals() and not callable(C): del C


def write_book(path, title, sheets):
    """Excel workbook: README sheet (title + list of sheets), then one sheet per panel in the given order."""
    readme = pd.DataFrame([(n, d, len(df)) for n, d, df in sheets], columns=["Sheet", "Content", "Rows"])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        readme.to_excel(xw, sheet_name="README", index=False, startrow=2)
        for n, d, df in sheets:
            df.to_excel(xw, sheet_name=str(n)[:31], index=False, startrow=2)
        wb = xw.book
        wb["README"]["A1"] = title; wb["README"]["A1"].font = Font(bold=True, size=12)
        for n, d, df in sheets:
            ws = wb[str(n)[:31]]; ws["A1"] = d; ws["A1"].font = Font(bold=True)
        for ws in wb.worksheets:
            for c in ws[3]: c.font = Font(bold=True)
            ws.freeze_panes = "A4"
            for col in ws.iter_cols(min_row=3, max_row=min(ws.max_row, 200)):
                w = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                ws.column_dimensions[col[0].column_letter].width = min(max(9, w + 2), 60)
        wb["README"].column_dimensions["B"].width = 110
    print("written:", path.name, "|", ", ".join(n for n, _, _ in sheets))


def or_row(f, term, label):
    lo, hi = np.exp(f.conf_int().loc[term])
    return {"comparison": label, "OR": np.exp(f.params[term]), "CI_lo": lo, "CI_hi": hi, "p": f.pvalues[term], "n": int(f.nobs)}


def mean_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    b = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return x.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)


M = o[o["is_mg"].astype(bool)].copy()
M["spp1"] = (M["Spp1_n"] >= 1).astype(int); M["age_mo"] = M["age_mo"].astype(float)
M["ctx2"] = np.where(M["ctx_bin"].isin(["Border", "Fibre"]), "Fibre / border", "Grey")
W = M[M["genotype"] == "WT"].copy(); A_WT = W["age_mo"].max(); W["aged"] = (W["age_mo"] == A_WT).astype(int)
T = M[M["genotype"] != "WT"].copy(); A_TG = T["age_mo"].max()
mic_cols = ["genotype", "age_mo", "x_centroid", "y_centroid", "ctx_bin", "Spp1_n", "n_astro25", "d_vlmc"]
summ = pd.DataFrame(summary) if "summary" in globals() else pd.DataFrame()

# ---------------- Figure 2
s2A = W[[c for c in mic_cols if c in W]].rename(columns={"x_centroid": "x_um", "y_centroid": "y_um"})
s2B = (W.groupby(["age_mo", "ctx_bin"]).agg(microglia=("spp1", "size"), spp1_pos=("spp1", "sum")).reset_index()
         .assign(pct_spp1_pos=lambda t: 100 * t["spp1_pos"] / t["microglia"]))
s2B_m = pd.DataFrame([
    or_row(smf.glm("spp1 ~ aged + C(ctx_bin)", data=W, family=sm.families.Binomial()).fit(cov_type="HC1"), "aged",
           f"{A_WT:g} months vs younger, context as fixed effect"),
    or_row(smf.glm("spp1 ~ aged + C(ctx_bin)", data=W[W["d_vlmc"] > 30], family=sm.families.Binomial()).fit(cov_type="HC1"), "aged",
           f"{A_WT:g} months vs younger, excluding microglia within 30 µm of fibroblast-like cells")])
idg = ["P2ry12", "Tmem119", "Cx3cr1", "Hexb", "Cd63", "Ctsb", "Npc2", "Lyz2", "Itgax", "Gpnmb", "Cd74"]
cp, cn = f"WT {A_WT:g} Spp1+", f"WT {A_WT:g} Spp1−"
s2C = pd.DataFrame({"gene": [g for g in idg if g in det.index]})
s2C["Spp1+ detecting (%)"] = det.loc[s2C["gene"], cp].values; s2C["Spp1− detecting (%)"] = det.loc[s2C["gene"], cn].values
s2C["difference (percentage points)"] = s2C["Spp1+ detecting (%)"] - s2C["Spp1− detecting (%)"]
cc = [(cnt(cp, g), cnt(cn, g)) for g in s2C["gene"]]
s2C["Fisher p"] = [fisher_exact([[a1, b1], [a2, b2]])[1] for (a1, b1), (a2, b2) in cc]
s2C["n Spp1+"], s2C["n Spp1−"] = ns[cp], ns[cn]
t_old = W[W["aged"] == 1]
s2D = []
for cx_ in ["Fibre / border", "Grey"]:
    t = t_old[t_old["ctx2"] == cx_]; cont = t["n_astro25"] >= 1
    a = int((cont & (t["spp1"] == 1)).sum()); b = int((t["spp1"] == 1).sum()) - a
    c = int((cont & (t["spp1"] == 0)).sum()); d = int((t["spp1"] == 0).sum()) - c
    s2D.append({"context": cx_, "Spp1+ with astrocyte": a, "Spp1+ without": b, "Spp1+ contact (%)": 100 * a / max(a + b, 1),
                "Spp1− with astrocyte": c, "Spp1− without": d, "Spp1− contact (%)": 100 * c / max(c + d, 1),
                "Fisher p": fisher_exact([[a, b], [c, d]])[1]})
s2D = pd.DataFrame(s2D)
s2D_m = pd.DataFrame([or_row(smf.glm("spp1 ~ n_astro25 + C(ctx_bin)", data=t_old, family=sm.families.Binomial()).fit(cov_type="HC1"),
                             "n_astro25", f"WT {A_WT:g} months, per astrocyte neighbour, context as fixed effect")])
if len(summ): s2D_m = pd.concat([s2D_m, summ.assign(note="core statistics per animal (cell 6)")], ignore_index=True)
pm12 = pd.read_csv(TAB / "10x_12_transcript_level_per_microglia.csv", index_col=0)
tr12 = pd.read_csv(TAB / "10x_12_transcript_level_astro.csv")
pw_ = pm12[(pm12["genotype"] == "WT") & (pm12["age_mo"] == A_WT)].copy()
pw_["ctx2"] = np.where(pw_["ctx_bin"].isin(["Border", "Fibre"]), "Fibre / border", "Grey")
s2E1 = pd.DataFrame([dict(zip(["context", "Spp1", "mean", "CI_lo", "CI_hi"], [cx_, "Spp1+" if sp else "Spp1−",
                     *mean_ci(pw_.loc[(pw_["ctx2"] == cx_) & (pw_["spp1"] == sp), "cover"])]),
                     n=int(((pw_["ctx2"] == cx_) & (pw_["spp1"] == sp)).sum())) for cx_ in ["Fibre / border", "Grey"] for sp in [0, 1]])
s2E2 = tr12[(tr12["genotype"] == "WT") & (tr12["age"] == A_WT) & (tr12["variant"] == "all")].copy()

write_book(XLS / "Supplementary_Table_S1_source_data_Figure1.xlsx", "Supplementary Table S1. Source data of Figure 1",
           [("1A", "1A: strict microglia of the wild-type mice shown on the sections (coordinates in µm)", s2A),
            ("1B", "1B: Spp1+ microglia by local context and age", s2B),
            ("1B_model", "1B: age effect, binomial GLM with context as fixed effect, HC1", s2B_m),
            ("1C", f"1C: gene detection in Spp1+ vs Spp1− microglia at {A_WT:g} months after downsampling to 100 transcripts per cell", s2C),
            ("1D", f"1D: microglia with ≥1 astrocyte within 25 µm at {A_WT:g} months, Fisher's exact test per context", s2D),
            ("1D_model", "1D: OR per astrocyte neighbour, and core statistics per animal", s2D_m),
            ("1E_coverage", "1E left: homeostatic astrocyte transcripts 5 to 25 µm per microglia, mean and bootstrap 95% CI", s2E1),
            ("1E_genes", "1E right: coverage rate ratio and gene odds ratios relative to homeostatic astrocyte transcripts, Spp1+ vs Spp1−", s2E2)])

# ---------------- Figure 3
sel3 = M[((M["genotype"] == "WT") & (M["age_mo"] == A_WT)) | ((M["genotype"] != "WT") & (M["age_mo"] == A_TG))]
s3A = sel3[[c for c in mic_cols if c in sel3]].rename(columns={"x_centroid": "x_um", "y_centroid": "y_um"})
s3B = (M.groupby(["genotype", "age_mo", "ctx2"]).agg(microglia=("spp1", "size"), spp1_pos=("spp1", "sum")).reset_index()
         .assign(pct_spp1_pos=lambda t: 100 * t["spp1_pos"] / t["microglia"]))
pm12["tg"] = (pm12["genotype"] != "WT").astype(int)
dC_ = pm12[((pm12["tg"] == 0) & (pm12["age_mo"] == A_WT)) | ((pm12["tg"] == 1) & (pm12["age_mo"] == A_TG))].copy()
dC_["gfap_share"] = dC_["Gfap"] / (dC_["Gfap"] + dC_["cover"]).replace(0, np.nan)
s3C = pd.DataFrame([dict(zip(["genotype", "Spp1", "measure", "mean", "CI_lo", "CI_hi"],
                    [g, "Spp1+" if sp else "Spp1−", v, *mean_ci(dC_.loc[(dC_["genotype"] == g) & (dC_["spp1"] == sp), v])]))
                    for v in ["cover", "gfap_share"] for g in dC_["genotype"].unique() for sp in [0, 1]])
s3C_m = pd.DataFrame([{"measure": k, "interaction p (Spp1 state x genotype)": v} for k, v in P_INT.items()]) if "P_INT" in globals() else pd.DataFrame()
s3D = pd.DataFrame(idf, index=ID_ROWS, columns=gl).reset_index(names="group")
s3D_m = pd.DataFrame(idm, index=ID_ROWS, columns=gl).reset_index(names="group")
s3D_n = pd.DataFrame({"group": ID_ROWS, "microglia (≥100 transcripts)": [ns[r] for r in ID_ROWS]})
s3D_stats = pd.read_csv(TAB / "10xAD_08_identity_spp1pos_WT_vs_Tg.csv")
s3E = pd.concat([d.assign(genotype=g) for g, d in exF.items()], ignore_index=True)
sheets3 = [("4A", f"4A: strict microglia of the WT {A_WT:g}-month and TgCRND8 {A_TG:g}-month mice (coordinates in µm)", s3A),
           ("4B", "4B: Spp1+ microglia in white matter (fibre and border) vs grey matter, all ages and both genotypes", s3B),
           ("4C", "4C: homeostatic astrocyte transcripts and Gfap share (5 to 25 µm) around Spp1− and Spp1+ microglia, mean and bootstrap 95% CI", s3C),
           ("4C_model", "4C: interaction of Spp1 state and genotype (section x context fixed effects, HC1)", s3C_m),
           ("4D", "4D: astrocytes adjacent to Spp1+ vs Spp1− microglia: mean log-normalised expression, log2 fold change, permutation p within tissue type (5,000 permutations)", s3E),
           ("4E_detection", "4E: % of microglia detecting each gene after downsampling to 100 transcripts per cell", s3D),
           ("4E_expression", "4E: mean log expression after downsampling (dot colour, scaled per gene in the figure)", s3D_m),
           ("4E_n", "4E: microglia per group", s3D_n),
           ("4E_stats", "4E: Spp1+ microglia TgCRND8 17.9 vs WT 13.4 months, Fisher's exact test with BH correction", s3D_stats),
           ("4G", "4G: same measures in all cohorts. OR per astrocyte within 25 µm (Spp1+ vs Spp1−, tissue context as fixed effect; own cohort from Z1, CosMx per mouse from C6, CosMx pooled = inverse-variance fixed-effect meta-analysis); "
                  "Gfap log2 fold change in astrocytes adjacent to Spp1+ vs Spp1− microglia (own Z2, 10x 4D, CosMx C7, permutation p); "
                  "% Gfap+ astrocytes not adjacent to Spp1+ microglia in the young reference and at the stage (10x: same genotype at 2.5 months, Fisher's exact test; CosMx: young section on the same slide, white matter, paired t-test across mice)", G4TAB)]
if "G4TAB" not in globals(): sheets3 = sheets3[:-1]
write_book(XLS / "Supplementary_Table_S5_source_data_Figure4.xlsx", "Supplementary Table S5. Source data of Figure 4", sheets3)